## Correspondance avec les livrables (§9 du sujet)

| Livrable | Fichier / dossier |
|----------|-------------------|
| 1. Rapport scientifique | `rapport/rapport_scientifique.md` → PDF |
| 2. Code source commenté | `src/partie*.py` |
| 3. Notebook exécutable | ce fichier + `main.py` |
| 4. Annexe expérimentale | `annexe/` (généré par `py main.py`) |

Voir `LIVRABLES.md` à la racine du projet.

# Partie III – RNN, LSTM, GRU et Seq2Seq
**Projet Deep Learning — EMSI Casablanca 2025–2026**

**Dataset :** corpus parallèle **fra-eng** (ManyThings, téléchargement auto) — équivalent fra-eng du sujet EMSI  
**Tâches :** modélisation de séquences, comparaison RNN/LSTM/GRU, traduction Seq2Seq, décodage glouton vs beam search, métrique BLEU.

## 0. Dépendances

In [1]:
%pip install -q torch matplotlib sacrebleu

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Users\HP\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


## 1. Théorie – modèle de langage et perplexité

**Factorisation :** $P(x_1,\ldots,x_T) = \prod_t P(x_t \mid x_{<t})$

**Perplexité :** $\mathrm{PPL} = \exp\big(-\frac{1}{T}\sum_t \log P(x_t \mid x_{<t})\big)$ — plus bas = mieux.

**BPTT** : rétropropagation à travers le temps ; risque de gradients explosifs → **gradient clipping**.

In [2]:
import math
import random
import time
from collections import Counter

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

from torchtext.data.utils import get_tokenizer
from src.corpus import load_parallel_pairs
from src.vocab_utils import build_vocab, SimpleVocab

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device :", device)
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

SPECIALS = ["<pad>", "<bos>", "<eos>", "<unk>"]
PAD, BOS, EOS, UNK = SPECIALS
MAX_LEN = 25
BATCH_SIZE = 64
EMBED_DIM = 128
HIDDEN_DIM = 256

Device : cpu


## 2. Chargement et préparation du corpus fra-eng

In [3]:
tokenizer = get_tokenizer("basic_english")

MAX_TRAIN = 8000
MAX_VAL = 800
train_pairs, val_pairs = load_parallel_pairs(max_train=MAX_TRAIN, max_val=MAX_VAL)
print("Exemple :", train_pairs[0])

Corpus fra-eng : 8000 train, 800 val
Exemple : ('Go.', 'Va !')


In [4]:
src_vocab = build_vocab(tokenizer, train_pairs, lang_idx=0, specials=SPECIALS)
tgt_vocab = build_vocab(tokenizer, train_pairs, lang_idx=1, specials=SPECIALS)

def encode(tokens, vocab, add_bos_eos=False):
    ids = [vocab[t] for t in tokens]
    if add_bos_eos:
        return [vocab["<bos>"]] + ids + [vocab["<eos>"]]
    return ids

def decode(ids, vocab):
    itos = vocab.get_itos()
    words = []
    for i in ids:
        w = itos[i]
        if w == EOS:
            break
        if w not in (PAD, BOS):
            words.append(w)
    return " ".join(words)

In [5]:
class TranslationDataset(Dataset):
    def __init__(self, pairs, max_len=MAX_LEN):
        self.samples = []
        for src, tgt in pairs:
            src_ids = encode(tokenizer(src), src_vocab)[: max_len - 2]
            tgt_ids = encode(tokenizer(tgt), tgt_vocab)[: max_len - 2]
            self.samples.append(
                (
                    torch.tensor(src_ids, dtype=torch.long),
                    torch.tensor(tgt_ids, dtype=torch.long),
                )
            )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]


def collate_batch(batch):
    src_list, tgt_list = zip(*batch)
    src_pad = pad_sequence(src_list, batch_first=True, padding_value=src_vocab["<pad>"])
    tgt_in = [torch.tensor([tgt_vocab["<bos>"]] + t.tolist(), dtype=torch.long) for t in tgt_list]
    tgt_out = [torch.tensor(t.tolist() + [tgt_vocab["<eos>"]], dtype=torch.long) for t in tgt_list]
    tgt_in_pad = pad_sequence(tgt_in, batch_first=True, padding_value=tgt_vocab["<pad>"])
    tgt_out_pad = pad_sequence(tgt_out, batch_first=True, padding_value=tgt_vocab["<pad>"])
    return src_pad, tgt_in_pad, tgt_out_pad

train_ds = TranslationDataset(train_pairs)
val_ds = TranslationDataset(val_pairs)
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_batch)
val_dl = DataLoader(val_ds, batch_size=BATCH_SIZE, collate_fn=collate_batch)
len(src_vocab), len(tgt_vocab)

(1172, 1725)

## 3. Comparaison RNN / LSTM / GRU (encodeur seul – perplexité)

In [6]:
class RecurrentLM(nn.Module):
    def __init__(self, vocab_size, cell="gru", hidden=HIDDEN_DIM, embed=EMBED_DIM):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed, padding_idx=src_vocab["<pad>"])
        if cell == "rnn":
            self.rnn = nn.RNN(embed, hidden, batch_first=True)
        elif cell == "lstm":
            self.rnn = nn.LSTM(embed, hidden, batch_first=True)
        else:
            self.rnn = nn.GRU(embed, hidden, batch_first=True)
        self.fc = nn.Linear(hidden, vocab_size)
        self.cell = cell

    def forward(self, x):
        emb = self.embed(x)
        if self.cell == "lstm":
            out, _ = self.rnn(emb)
        else:
            out, _ = self.rnn(emb)
        return self.fc(out)


def masked_nll(logits, targets, pad_idx):
    loss = F.cross_entropy(
        logits.reshape(-1, logits.size(-1)),
        targets.reshape(-1),
        ignore_index=pad_idx,
        reduction="sum",
    )
    mask = targets != pad_idx
    n_tokens = mask.sum().item()
    return loss / max(n_tokens, 1), n_tokens


import torch.nn.functional as F

def train_lm(cell, epochs=5):
    model = RecurrentLM(len(src_vocab), cell=cell).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    losses = []
    for ep in range(1, epochs + 1):
        model.train()
        total_loss, n_tok = 0.0, 0
        for src, _, _ in train_dl:
            src = src.to(device)
            inp, tgt = src[:, :-1], src[:, 1:]
            logits = model(inp)
            loss, nt = masked_nll(logits, tgt, src_vocab["<pad>"])
            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            total_loss += loss.item() * nt
            n_tok += nt
        ppl = math.exp(total_loss / n_tok)
        losses.append(ppl)
        print(f"{cell.upper()} epoch {ep} | perplexité train ≈ {ppl:.2f}")
    return model, losses

In [7]:
lm_results = {}
for cell in ["rnn", "lstm", "gru"]:
    _, losses = train_lm(cell, epochs=4)
    lm_results[cell] = losses

plt.figure(figsize=(7, 4))
for cell, losses in lm_results.items():
    plt.plot(losses, marker="o", label=cell.upper())
plt.ylabel("Perplexité")
plt.xlabel("Époque")
plt.title("RNN vs LSTM vs GRU – langage source EN")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

RNN epoch 1 | perplexité train ≈ 46.58


RNN epoch 2 | perplexité train ≈ 14.02


RNN epoch 3 | perplexité train ≈ 9.92


RNN epoch 4 | perplexité train ≈ 8.13


LSTM epoch 1 | perplexité train ≈ 72.90


...

GRU epoch 1 | perplexité train ≈ 61.32


GRU epoch 2 | perplexité train ≈ 15.96


GRU epoch 3 | perplexité train ≈ 10.50


GRU epoch 4 | perplexité train ≈ 8.22


C:\Users\HP\AppData\Local\Temp\ipykernel_18088\3825322595.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Seq2Seq encodeur–décodeur (GRU) + teacher forcing

In [8]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, embed=EMBED_DIM, hidden=HIDDEN_DIM):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed, padding_idx=src_vocab["<pad>"])
        self.rnn = nn.GRU(embed, hidden, batch_first=True)

    def forward(self, src):
        emb = self.embed(src)
        outputs, hidden = self.rnn(emb)
        return outputs, hidden


class Decoder(nn.Module):
    def __init__(self, vocab_size, embed=EMBED_DIM, hidden=HIDDEN_DIM):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed, padding_idx=tgt_vocab["<pad>"])
        self.rnn = nn.GRU(embed, hidden, batch_first=True)
        self.fc = nn.Linear(hidden, vocab_size)

    def forward(self, tgt_in, hidden):
        emb = self.embed(tgt_in)
        out, hidden = self.rnn(emb, hidden)
        return self.fc(out), hidden


class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, tgt_in):
        _, hidden = self.encoder(src)
        logits, _ = self.decoder(tgt_in, hidden)
        return logits

In [9]:
def train_seq2seq(epochs=8):
    model = Seq2Seq(Encoder(len(src_vocab)), Decoder(len(tgt_vocab))).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    train_losses = []
    for ep in range(1, epochs + 1):
        model.train()
        total, n_tok = 0.0, 0
        t0 = time.time()
        for src, tgt_in, tgt_out in train_dl:
            src, tgt_in, tgt_out = src.to(device), tgt_in.to(device), tgt_out.to(device)
            logits = model(src, tgt_in)
            loss, nt = masked_nll(logits, tgt_out, tgt_vocab["<pad>"])
            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # BPTT + clipping
            opt.step()
            total += loss.item() * nt
            n_tok += nt
        train_losses.append(total / n_tok)
        print(f"Seq2Seq epoch {ep} | loss/token={train_losses[-1]:.4f} | {time.time()-t0:.1f}s")
    return model, train_losses

seq2seq_model, s2s_losses = train_seq2seq(epochs=8)
plt.plot(s2s_losses)
plt.title("Perte Seq2Seq (token moyen)")
plt.xlabel("Époque")
plt.show()

Seq2Seq epoch 1 | loss/token=3.5968 | 7.4s


Seq2Seq epoch 2 | loss/token=2.3065 | 8.4s


Seq2Seq epoch 3 | loss/token=1.8601 | 9.0s


Seq2Seq epoch 4 | loss/token=1.5573 | 8.1s


Seq2Seq epoch 5 | loss/token=1.3253 | 7.7s


Seq2Seq epoch 6 | loss/token=1.1360 | 8.8s


Seq2Seq epoch 7 | loss/token=0.9794 | 8.1s


Seq2Seq epoch 8 | loss/token=0.8478 | 7.4s


C:\Users\HP\AppData\Local\Temp\ipykernel_18088\2730517988.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Décodage glouton vs beam search

In [10]:
def translate_sentence(model, sentence, max_len=MAX_LEN, method="greedy", beam_width=3):
    model.eval()
    tokens = tokenizer(sentence)
    src = torch.tensor([encode(tokens, src_vocab)], device=device)
    with torch.no_grad():
        enc_out, hidden = model.encoder(src)

    generated = [tgt_vocab["<bos>"]]

    if method == "greedy":
        dec_in = torch.tensor([[tgt_vocab["<bos>"]]], device=device)
        for _ in range(max_len):
            logits, hidden = model.decoder(dec_in, hidden)
            next_id = logits[:, -1, :].argmax(-1).item()
            generated.append(next_id)
            if next_id == tgt_vocab["<eos>"]:
                break
            dec_in = torch.tensor([[next_id]], device=device)
        return decode(generated, tgt_vocab)

    # Beam search simplifié
    beams = [([tgt_vocab["<bos>"]], hidden, 0.0)]
    completed = []
    for _ in range(max_len):
        new_beams = []
        for seq, h, score in beams:
            if seq[-1] == tgt_vocab["<eos>"]:
                completed.append((seq, score))
                continue
            dec_in = torch.tensor([[seq[-1]]], device=device)
            logits, new_h = model.decoder(dec_in, h)
            log_probs = F.log_softmax(logits[:, -1, :], dim=-1).squeeze(0)
            topk = log_probs.topk(beam_width)
            for i in range(topk.values.size(0)):
                nid = topk.indices[i].item()
                new_beams.append((seq + [nid], new_h, score + topk.values[i].item()))
        beams = sorted(new_beams, key=lambda x: x[2] / len(x[0]), reverse=True)[:beam_width]
        if all(s[-1] == tgt_vocab["<eos>"] for s, _, _ in beams):
            break
    best = beams[0][0] if beams else generated
    return decode(best, tgt_vocab)

In [11]:
examples = [
    "a group of people are sitting on a bench",
    "two young boys playing with a ball",
    "the man is wearing a red shirt",
]
for sent in examples:
    g = translate_sentence(seq2seq_model, sent, method="greedy")
    b = translate_sentence(seq2seq_model, sent, method="beam", beam_width=5)
    print("EN :", sent)
    print("Greedy DE :", g)
    print("Beam DE   :", b)
    print("-" * 60)

EN : a group of people are sitting on a bench
Greedy DE : quelle <unk> !
Beam DE   : c ' est un <unk> !
------------------------------------------------------------
EN : two young boys playing with a ball
Greedy DE : <unk> un <unk> !
Beam DE   : <unk> d ' un <unk> de <unk> ton âge ! <unk> ! <unk> .
------------------------------------------------------------
EN : the man is wearing a red shirt
Greedy DE : c ' est un <unk> .
Beam DE   : c ' est un <unk> de l ' heure de <unk> .
------------------------------------------------------------


## 6. Évaluation BLEU (validation)

In [12]:
try:
    import sacrebleu
except ImportError:
    sacrebleu = None

def compute_bleu(model, pairs, n=200, method="greedy"):
    refs, hyps = [], []
    for i, (src, tgt) in enumerate(pairs[:n]):
        pred = translate_sentence(model, src, method=method, beam_width=5)
        ref = " ".join(tokenizer(tgt))
        hyps.append(pred)
        refs.append([ref])
    if sacrebleu:
        return sacrebleu.corpus_bleu(hyps, refs).score
    # fallback simpliste : exact match ratio
    exact = sum(h == r[0] for h, r in zip(hyps, refs)) / len(refs)
    return exact

bleu_greedy = compute_bleu(seq2seq_model, val_pairs, n=150, method="greedy")
bleu_beam = compute_bleu(seq2seq_model, val_pairs, n=150, method="beam")
print(f"BLEU (greedy, n=150) : {bleu_greedy:.2f}")
print(f"BLEU (beam, n=150)   : {bleu_beam:.2f}")

BLEU (greedy, n=150) : 15.85
BLEU (beam, n=150)   : 16.51


## 7. Question de synthèse – Partie III

**Question :** Dans quelle mesure les architectures récurrentes modélisent-elles efficacement une séquence réelle ? Justifier le passage RNN → LSTM/GRU → Seq2Seq.

**Éléments pour votre rapport :**

1. **Modèle de langage** : factorisation par la règle de chaîne ; perplexité pour comparer RNN/LSTM/GRU.
2. **Stabilité** : LSTM/GRU gèrent mieux les dépendances longues ; clipping indispensable sur RNN simple.
3. **Seq2Seq** : encodeur résume la source ; décodeur génère la cible mot à mot ; teacher forcing stabilise l'entraînement.
4. **Décodage** : glouton rapide mais sous-optimal ; beam search explore plusieurs hypothèses → BLEU souvent meilleur.
5. **Limites** : compression du contexte dans un seul vecteur ; pas d'attention → performances plafonnent vs Transformers.

---

## 8. Question transversale (rapport final)

Relier MLP (géométrie tabulaire), CNN (localité spatiale) et RNN/Seq2Seq (dépendances temporelles) : le paradigme supervisé reste identique (perte + gradient), mais l'**inductive bias** architectural doit suivre la structure des données.

## 9. Export vers l'annexe expérimentale

In [13]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
from src import partie3_seq2seq
metrics = partie3_seq2seq.run(export=True, lm_epochs=3, s2s_epochs=5)
print("→ annexe/partie3/")

Corpus fra-eng : 8000 train, 800 val


  LM rnn epoch 1/3 perplexite=46.58


  LM rnn epoch 2/3 perplexite=14.02


  LM rnn epoch 3/3 perplexite=9.92


  LM lstm epoch 1/3 perplexite=74.04


...

  Seq2Seq epoch 3/5 loss/token=1.8511 (7s)


  Seq2Seq epoch 4/5 loss/token=1.5561 (8s)


  Seq2Seq epoch 5/5 loss/token=1.3295 (8s)


  BLEU greedy=26.27 beam=17.97


→ annexe/partie3/
